In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv(r"C:\Users\Administrador\Downloads\coaster_db.csv")
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df

# Proceso de limpiado de Datos

### Cambio valores de altura y velocidad a numericos quitando las unidades

In [ ]:
for col in ['height', 'speed']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.extract(r'(\d+\.?\d*)')
        df[col] = pd.to_numeric(df[col], errors='coerce')


In [ ]:
def to_meters(value, unit):
    if pd.isna(value) or pd.isna(unit):
        return np.nan
    if unit == "ft":
        return round(value * 0.3048,2)
    elif unit == "m":
        return value
    else:
        return np.nan  # por si aparece otra cosa inesperada

# Crear nueva columna
df["height_m"] = df.apply(lambda row: to_meters(row["height_value"], row["height_unit"]), axis=1)
df

### Consulto y filtro filas y columnas no validas

In [ ]:
print(f"Tenemos {df.duplicated().sum()} filas duplicadas")

In [ ]:
# Heatmap de nulos
plt.figure(figsize=(12,6))
sns.heatmap(df.isnull(), cbar=False)
plt.title("Mapa de valores nulos")
plt.show()

In [ ]:
df_aeliminar = df.isnull().sum().sort_values(ascending=False).head(10)
print(df_aeliminar)
df_Limpio = df.drop(columns = list(df_aeliminar.index))
registrosMalos = (df.isnull().sum(axis=1) / df.shape[1]).sort_values(ascending=False).head(10).round(2)
df_Limpio = df_Limpio.drop(index=registrosMalos.index)
df_Limpio = df_Limpio.drop(index= 905)


### Mapa de valores nulos dsp de borar top 10 filas y columnas con mas nulos

In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(df_Limpio.isnull(), cbar=False)
plt.title("Mapa de valores nulos")
plt.show()

### Mapa de valores nulos despues de borrar columnas con mas de 50% de valores nulos / Ultimo filtrado

In [ ]:
column_nulos = df_Limpio.isnull().mean() > 0.5
print(f"Columnas eliminadas por nulos altos: {column_nulos.sum()}")
df_Limpio = df_Limpio.drop(columns=df_Limpio.columns[column_nulos])

plt.figure(figsize=(12,6))
sns.heatmap(df_Limpio.isnull(), cbar=False)
plt.title("Mapa de valores nulos")
plt.show()

### Despues filtro y mantengo unicamente las columnas que utilizare en las visualizaciones para manener ordenado el dataset

In [ ]:
df_Limpio = df_Limpio[['coaster_name', 'height_m', 'speed_mph','year_introduced', 'inversions']]
cols_sucio = set(col.strip().lower() for col in df.columns)
cols_limpio = set(col.strip().lower() for col in df_Limpio.columns)
cols_eliminadas = cols_sucio - cols_limpio
print("Columnas eliminadas:")
for col in cols_eliminadas:
    print(col)
df_Limpio

# Graficado de Datos

In [ ]:

if 'height_m' in df_Limpio.columns:
    plt.hist(df_Limpio['height_m'].dropna(), bins=50, edgecolor='k')
    plt.title("Distribución de alturas")
    plt.xlabel("Altura en metros")
    valor_marcar = df_Limpio['height_m'].mean()
    plt.xticks(rotation = 0)
    plt.axvline(x=valor_marcar, color='red', linestyle='--', linewidth=2, label=f"Altiura Promedio = {valor_marcar}     ")
    plt.text(
        0.95, 0.95,
        f"---- el promedio de alturas es de {valor_marcar:.1f} m",
        transform=plt.gca().transAxes,
        ha="right", va="top",
        fontsize=10, color="red"
    )
    plt.show()
df_Limpio['height_m'].describe()

In [ ]:
q1 = df_Limpio['speed_mph'].quantile(0.25)
q3 = df_Limpio['speed_mph'].quantile(0.75)
irq = q3 - q1
if 'speed_mph' in df_Limpio.columns:
    sns.boxplot(x=df_Limpio['speed_mph'], showfliers= False)
    plt.title("Boxplot de velocidades en mph")
    plt.xticks([( q1 - 1.5 * (irq)),df_Limpio['speed_mph'].median(),( q3 + 1.5 * (irq))])
    plt.text(
    0.95, 0.95,
    f"Máximo real (atipico) : {df_Limpio['speed_mph'].max()} m/hrs",
    transform=plt.gca().transAxes,
    ha="right", va="top",
    fontsize=10, color="yellow"
)
    plt.show()
vel_anormal_Mayor = df_Limpio.loc[df_Limpio['speed_mph'] > ( q3 + 1.5 * (irq)), ['coaster_name', 'speed_mph']]
vel_anormal_Menor = df_Limpio.loc[df_Limpio['speed_mph'] < ( q1 - 1.5 * (irq)), ['coaster_name', 'speed_mph']]
vel_anormal = pd.concat([vel_anormal_Menor, vel_anormal_Mayor], axis=0)
print('Montanas rusas con velocidad anormal (de menor a mayor)')
vel_anormal.sort_values(by='speed_mph', ascending=True, inplace=True)
vel_anormal


In [ ]:
coasters_por_ano = df_Limpio['year_introduced'].value_counts().sort_index()
plt.figure(figsize=(12, 6))
promedio_coasters = coasters_por_ano.mean()
coasters_por_ano.plot(kind='line', color='red', label='Coasters por año')
plt.axhline(y=promedio_coasters, color='cyan', linestyle='-', linewidth=2, label=f'Promedio = {promedio_coasters:.2f}')
plt.title('Evolución de Montañas Rusas Introducidas por Año')
plt.xlabel('Año')
plt.ylabel('Número de Coasters')
plt.text(
    0.47, 0.95,
    f"La cantidad promedio de montanas introducidas es de {coasters_por_ano.mean():.1f} por ano ",
    transform=plt.gca().transAxes,
    ha="right", va="top",
    fontsize=10, color="cyan"
)
plt.show()

In [ ]:
conteo = df.groupby("year_introduced")["inversions"].mean()
plt.figure(figsize=(12, 6))
plt.plot(conteo.index, conteo.values)
plt.xlabel("Año")
plt.ylabel("Promedio de inversiones (rulos) por año")
plt.axhline(y=conteo.mean(), color='cyan', linestyle='-', linewidth=2, label=f'Promedio = {promedio_coasters:.2f}')
plt.title("Cantidad de inversiones por año")
plt.grid(True)
plt.show()